In [8]:
# importing needed packages
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [9]:
# read in the cleaned CSV with data from central valley stations
path = 'E:/Central Valley Fog/CV_rows_codes_incl.csv'
if os.path.exists(path):
    CV_rows = pd.read_csv(path, low_memory = False)
else:
    path = '/Volumes/disk1/Central Valley Fog/CV_rows_codes_incl.csv'
    CV_rows = pd.read_csv(path, low_memory = False)

print(CV_rows)

           STATION_ID  LATITUDE  LONGITUDE                 DATE REPORT_TYPE  \
0         USM00074917   35.1170  -119.3000  1941-12-01 00:00:00         SAO   
1         USM00074917   35.1170  -119.3000  1941-12-01 01:00:00         SAO   
2         USM00074917   35.1170  -119.3000  1941-12-01 02:00:00         SAO   
3         USM00074917   35.1170  -119.3000  1941-12-01 03:00:00         SAO   
4         USM00074917   35.1170  -119.3000  1941-12-01 04:00:00         SAO   
...               ...       ...        ...                  ...         ...   
12877763  USW00024257   40.5175  -122.2986  2026-06-20 04:53:00       FM-15   
12877764  USW00024257   40.5175  -122.2986  2026-06-20 05:53:00       FM-15   
12877765  USW00024257   40.5175  -122.2986  2026-06-20 06:53:00       FM-15   
12877766  USW00024257   40.5175  -122.2986  2026-06-20 07:53:00       FM-15   
12877767  USW00024257   40.5175  -122.2986  2026-06-20 08:53:00       FM-15   

          HourlyPrecipitation HourlyPresentWeatherT

In [12]:
# filtering based on precipiation to identify fog conditions
# filtering visibility threshold to identify outliers
CV_fog_rows = CV_rows[
    (CV_rows["HourlyPrecipitation"] < 0.03)
    & (CV_rows["HourlyVisibility"] < 17.000)
]

print("CV_fog_rows:", CV_fog_rows.shape) 

CV_fog_rows: (10429072, 15)


In [13]:
# Unique codes in REPORT_TYPE
report_type_codes = sorted(CV_fog_rows["REPORT_TYPE"].dropna().astype(str).unique())

# Unique sky-condition codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_sky_codes = sorted(
    CV_fog_rows["HourlySkyConditions"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

#find unique dailyweather codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_weather_codes = sorted(
    CV_fog_rows["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"REPORT_TYPE unique codes ({len(report_type_codes)}):")
print(report_type_codes)

print(f"\nHourlySkyConditions unique codes ({len(hourly_sky_codes)}):")
print(hourly_sky_codes)

print(f"\nHourlyPresentWeatherType unique codes ({len(hourly_weather_codes)}):")
print(hourly_weather_codes)

REPORT_TYPE unique codes (10):
['AUTO', 'FM-12', 'FM-15', 'FM-16', 'SAO', 'SAOSP', 'SMARS', 'SY-MT', 'SYSA', 'WBO_F']

HourlySkyConditions unique codes (5):
['BKN', 'CLR', 'FEW', 'OVC', 'SCT']

HourlyPresentWeatherType unique codes (28):
['BCFG', 'BLDU', 'BR', 'DS', 'DU', 'DZ', 'FC', 'FG', 'FU', 'FZFG', 'FZRA', 'GR', 'GS', 'HZ', 'MIBR', 'MIFG', 'PL', 'PRFG', 'RA', 'SHRA', 'SN', 'SQ', 'SS', 'TSRA', 'UP', 'VCBLDU', 'VCFG', 'VCSHRA']


REPORT_TYPE unique codes + meanings:
'AUTO' 
'FM-12' 
'FM-15' 
'FM-16' 
'SAO' 
'SAOSP' 
'SMARS' 
'SY-MT' 
'SYSA' 
'WBO_F' 

HourlySkyConditions unique codes + meanings:
'BKN' - broken clouds
'CLR' - clear 
'FEW' - few clouds
'OVC' - overcast
'SCT' - scattered clouds

HourlyPresentWeatherConditions unique codes + meanings:

'BCFG' - patches fog
'BLDU' - blowing widespread dust
'BR' - mist
'DS' - dust storm
'DU' - widespread dust
'DZ' drizzle
'FC' - funnel cloud, waterspout, or tornado
'FG' - fog
'FU' - smoke
'FZFG' - freezing fog
'FZRA' - freezing rain
'GR' - hail
'GS' - small hail and/or snow pellets
'HZ' - haze
'MIBR' - shallow
'MIFG' - shallow
'PL' - ice pellets
'PRFG' - partial fog
'RA' - rain
'SHRA' - showers rain
'SN' - snow
'SQ' - squalls
'SS' - sandstorm
'TSRA' - thunderstorm rain
'UP'- unknown precipitation 
'VCBLDU' - vicinity blowing widespread dust
'VCFG' - vicinity fog
'VCSHRA' - vicinity showers rain
